# Fine-tuning Hebrew TrOCR (ViT encoder + DictaBERT decoder)

Interactive version of `train.py`. Fine-tunes on the handwritten dataset
`cyttic/trocr-hebrew-human` (document-level train/test split).

**Target hardware:** single L4 (24 GB, bf16). On a Turing card (RTX 2080) set
`PRECISION = "fp16"` further down — bf16 is unsupported there.

> Reminder: this human-only run is a **pipeline smoke-test + baseline**. The
> dataset is ~280 writers copying the same ~127 sentences, so a good test CER
> means "reads new handwriting of known sentences" — it will *not* generalize
> to unseen Hebrew words. That needs the synthetic-pretraining stage.

## 1. Setup

This notebook is **self-contained** — it downloads the model and dataset
from HuggingFace automatically. No other files needed; just run top to bottom.

In [ ]:
# Run once on a fresh VM. Comment out if deps are already installed.
!pip install -q torch transformers datasets accelerate jiwer pillow matplotlib

In [ ]:
import torch
import jiwer
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    VisionEncoderDecoderModel,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print("gpu   :", torch.cuda.get_device_name(0))
    print("vram  :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
    print("bf16  :", torch.cuda.is_bf16_supported())

In [ ]:
# --- HebrewBlockProcessor (inlined so this notebook is self-contained) ---
from PIL import Image, ImageOps
import numpy as np

class HebrewBlockProcessor:
    """Mirror (RTL->LTR) -> resize to 64px height -> tile into a 384x384 ViT container."""
    TARGET_HEIGHT = 64
    CONTAINER_SIZE = 384
    IMAGE_MEAN = [0.5, 0.5, 0.5]
    IMAGE_STD = [0.5, 0.5, 0.5]

    def __call__(self, images, return_tensors="pt"):
        if not isinstance(images, list):
            images = [images]
        pixel_values = torch.stack([self._process(img) for img in images])
        return {"pixel_values": pixel_values}

    def _process(self, image):
        image = image.convert("RGB")
        image = ImageOps.mirror(image)
        w, h = image.size
        new_w = max(1, round(w * self.TARGET_HEIGHT / h))
        image = image.resize((new_w, self.TARGET_HEIGHT), Image.LANCZOS)
        container = Image.new("RGB", (self.CONTAINER_SIZE, self.CONTAINER_SIZE), (255, 255, 255))
        img_arr = np.array(image)
        src_x, dest_x, dest_y = 0, 0, 0
        while src_x < new_w and dest_y < self.CONTAINER_SIZE:
            chunk_w = min(new_w - src_x, self.CONTAINER_SIZE - dest_x)
            chunk = Image.fromarray(img_arr[:, src_x:src_x + chunk_w])
            container.paste(chunk, (dest_x, dest_y))
            src_x += chunk_w
            dest_x += chunk_w
            if dest_x >= self.CONTAINER_SIZE:
                dest_x = 0
                dest_y += self.TARGET_HEIGHT
        t = torch.tensor(np.array(container), dtype=torch.float32).permute(2, 0, 1) / 255.0
        mean = torch.tensor(self.IMAGE_MEAN).view(3, 1, 1)
        std = torch.tensor(self.IMAGE_STD).view(3, 1, 1)
        return (t - mean) / std


## 2. Config

Knobs for the run. For a fast first experiment keep `ENCODER_FROZEN = True`.

In [ ]:
MODEL_ID          = "cyttic/trocr-hebrew-untrained"
DATASET_ID        = "cyttic/trocr-hebrew-human"
OUTPUT_DIR        = "output/trocr-hebrew-human"

EPOCHS            = 10
BATCH_SIZE        = 8
GRAD_ACCUM        = 1
LR                = 5e-5
MAX_TARGET_LENGTH = 128
NUM_WORKERS       = 4
PRECISION         = "bf16"   # "bf16" (L4) | "fp16" (RTX 2080) | "no"
ENCODER_FROZEN    = True
MAX_STEPS         = -1       # set e.g. 50 for a quick smoke test

## 3. Model, tokenizer, processor

In [ ]:
model     = VisionEncoderDecoderModel.from_pretrained(MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
processor = HebrewBlockProcessor()

# generation_config takes priority over model.config in recent transformers,
# so set the special tokens on it explicitly or generate() fails during eval.
model.generation_config.decoder_start_token_id = tokenizer.cls_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.sep_token_id
model.generation_config.max_new_tokens = None   # use max_length; silences the warning

if ENCODER_FROZEN:
    for p in model.encoder.parameters():
        p.requires_grad = False

n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"params total    : {n_total/1e6:.1f}M")
print(f"params trainable: {n_train/1e6:.1f}M")

## 4. Dataset

In [ ]:
ds = load_dataset(DATASET_ID)
ds

## 5. Sanity checks

Confirm zero document overlap, and *see* what `HebrewBlockProcessor` does:
mirror (RTL→LTR) → resize to 64px → tile into the 384×384 ViT container.

In [ ]:
tr = set(ds["train"]["source_doc"])
te = set(ds["test"]["source_doc"])
print(f"train docs: {len(tr)} | test docs: {len(te)} | OVERLAP: {len(tr & te)} (must be 0)")

In [ ]:
sample = ds["train"][1]
img = sample["image"].convert("RGB")
print("text:", sample["text"])

# what the model actually sees (denormalised back to [0,1] for display)
pv = processor([img])["pixel_values"][0]
shown = (pv * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].imshow(img);   ax[0].set_title("raw line crop (64px high)"); ax[0].axis("off")
ax[1].imshow(shown); ax[1].set_title("after HebrewBlockProcessor (mirrored + tiled 384x384)"); ax[1].axis("off")
plt.tight_layout(); plt.show()

## 6. Collator + metrics (CER / WER)

In [ ]:
def collate(batch):
    images = [ex["image"].convert("RGB") for ex in batch]
    texts  = [ex["text"] for ex in batch]
    pixel_values = processor(images)["pixel_values"]
    labels = tokenizer(
        texts, padding="longest", truncation=True,
        max_length=MAX_TARGET_LENGTH, return_tensors="pt",
    ).input_ids
    labels[labels == tokenizer.pad_token_id] = -100
    return {"pixel_values": pixel_values, "labels": labels}


def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    # generated preds AND labels can hold -100 padding -> invalid for decode
    pred_ids  = np.where(pred_ids  < 0, tokenizer.pad_token_id, pred_ids)
    label_ids = np.where(label_ids < 0, tokenizer.pad_token_id, label_ids)
    pred_str  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"cer": jiwer.cer(label_str, pred_str),
            "wer": jiwer.wer(label_str, pred_str)}

## 7. Train

Greedy decoding is used for the per-epoch eval (fast); a proper beam-search
eval is run once at the end.

In [ ]:
targs = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS,
    bf16=(PRECISION == "bf16"),
    fp16=(PRECISION == "fp16"),
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    dataloader_num_workers=NUM_WORKERS,
    remove_unused_columns=False,   # keep image/text for the custom collator
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=targs,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    data_collator=collate,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

## 8. Final evaluation (beam search) — the "real" number

In [ ]:
metrics = trainer.evaluate(num_beams=4, max_length=MAX_TARGET_LENGTH, metric_key_prefix="final")
print(f"FINAL CER: {metrics['final_cer']:.4f}")
print(f"FINAL WER: {metrics['final_wer']:.4f}")

## 9. Look at predictions

Qualitative check on a few held-out test lines.

In [ ]:
model.eval()
n = 6
fig, axes = plt.subplots(n, 1, figsize=(10, 2.2 * n))
for ax, ex in zip(axes, ds["test"].select(range(n))):
    img = ex["image"].convert("RGB")
    pv = processor([img])["pixel_values"].to(model.device)
    with torch.no_grad():
        ids = model.generate(pv, num_beams=4, max_new_tokens=MAX_TARGET_LENGTH)
    pred = tokenizer.batch_decode(ids, skip_special_tokens=True)[0]
    ax.imshow(img); ax.axis("off")
    ax.set_title(f"GT  : {ex['text']}\nPRED: {pred}", loc="left", fontsize=9)
plt.tight_layout(); plt.show()

## 10. Save / push the model

In [ ]:
best_dir = f"{OUTPUT_DIR}/best"
trainer.save_model(best_dir)
tokenizer.save_pretrained(best_dir)
print("saved ->", best_dir)

# Optional: push to the Hub (run `hf auth login` first)
# model.push_to_hub("cyttic/trocr-hebrew-human-finetuned")
# tokenizer.push_to_hub("cyttic/trocr-hebrew-human-finetuned")